> 랭체인 공식 문서 Tools: <https://docs.langchain.com/oss/python/langchain/tools>

### 도구 호출 에이전트(Tool Calling Agent)

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
model.invoke([HumanMessage("부산은 지금 몇시야?")])

AIMessage(content='현재 부산의 시각은 **[현재 시간]** 입니다.\n\n예를 들어, 지금이 2023년 10월 27일 오후 3시 45분이라면, 이렇게 답변해 드릴 수 있습니다:\n\n**"현재 부산의 시각은 오후 3시 45분입니다. (오늘 금요일, 10월 27일입니다.) 한국 표준시 (KST) 기준입니다."**\n\n정확한 현재 시각을 확인하려면, 이 답변을 생성하는 시점의 실시간 정보를 참조해야 합니다.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--863d2247-5f1e-493f-800d-6421673e068a-0', usage_metadata={'input_tokens': 8, 'output_tokens': 784, 'total_tokens': 792, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 661}})

#### ZoneInfo 간단 사용법

In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo # uv add tzdata

# 1. ZoneInfo 객체 생성
seoul_tz = ZoneInfo("Asia/Seoul") # 해당 지역의 시간대 정보 (오프셋, 일광 절약 시간제 등)
newyork_tz = ZoneInfo("America/New_York")
print(seoul_tz)
print(newyork_tz)


Asia/Seoul
America/New_York


In [4]:
type(seoul_tz)

zoneinfo.ZoneInfo

In [5]:
# 2. 현재 시간 가져오기 (시간대 정보 적용)
now_seoul = datetime.now(tz=seoul_tz)
now_newyork = datetime.now(tz=newyork_tz)

print(f"서울 현재 시간: {now_seoul}")
print(f"뉴욕 현재 시간: {now_newyork}")

서울 현재 시간: 2026-04-05 07:22:01.899078+09:00
뉴욕 현재 시간: 2026-04-04 18:22:01.899078-04:00


#### 도구 생성

In [6]:
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    target_timezone = ZoneInfo(timezone)
    now = datetime.now(target_timezone).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [7]:
from langchain.agents import create_agent

# 도구들을 tools 리스트에 추가
tools = [get_current_time,]

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."
)

In [8]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]},
)

Asia/Seoul (부산) 현재시각 2026-04-05 07:22:03 


In [9]:
result

{'messages': [HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}, id='d7810604-a226-4184-aee4-27b07c8a1a36'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'07400a75-6510-4d7b-bd9c-ea432fe9d24f': 'CvsDAb4+9vsAEY2unlj5XuwKjOouMJ3MbmS1TAR/jTmC/9zbrOLal4B+ZZqNsT7/pJjjLCCvxtIXdDTzh6bO4odK8ykP5ZlTHX5PlK0hKaN1g8iUApZUFFT2M1a9Uh8Wc/YKu45XZyAwc7ofguWZHtD6UtFZ0SDHpJ3sIlYzmqb8E/S3Ubf4UCymrqJDNiD6rlRecYDMrrm4YGIhMjuWFF9RXn04IEcntISLtVNhSvwWz/V9SB1j/cUImXsFxTCVTik4om/XF8EjCV/6YZ1TYX8+q4+tjxSb2mzuA+PeztesXcxXNnFuO7n1pV26OXro4hwejtz/kkParv2+VFQGnkaekiiVSl09NNJck1CXzg823KUMQpEJVtup0bAuaE8/5VESbX3XRWtG7a4Hxtp9pPO3ujxPrGCror03Vp9IpD0+Ui6gQnSGGsZlRwMM11rNty9I30aCLE0oAbkt8JFlHg2ZLUi7VwQbc+iIwUHpWB1q2fEQVrPRk0GQg/XH6sGNgHRpPoN3i5zmUPwWqPmvl43HJ5RVMqA+U4D0m//1/hAoXFqMNsvqFLKevrvbroiApgdqlkChM+FLKt1M5y9ujgTiVoKOcfTqvs